# Waste Type Identification — 08b: controllo del *confound* di geo_rrc


## 0. Configurazione
Da eseguire due volte, una con `HARD_LEVEL='mild'` e una con `HARD_LEVEL='moderate'`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE              = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')                              # copia locale
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
RESULTS_DIR       = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
NUM_CLASSES = 8
SEED        = 1234
HARD_LEVEL  = 'mild'                                                            # 'mild' oppure 'moderate'
EVAL_BATCH  = 128

CLASS_NAMES = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']

CANDIDATES = {
    'baseline (CE)'       : 'resnet18_baseline.pth',
    'light'               : 'resnet18_aug_light.pth',
    'geo_rrc (light+RRC)' : 'resnet18_geo_rrc.pth',
    'weighted_loss'       : 'resnet18_imbalance_weighted_loss.pth',
    'strong'              : 'resnet18_aug_strong.pth',
    'sampler'             : 'resnet18_imbalance_sampler.pth',
    'geo_rrc+cutmix'      : 'resnet18_geo_rrc_cutmix.pth',
}
REAL_CANDIDATES = {'light', 'geo_rrc (light+RRC)', 'weighted_loss'}

print('HARD_LEVEL =', HARD_LEVEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HARD_LEVEL = mild


## 1. Copia locale del dataset

In [ ]:
import shutil, time
if not DATASET_DIR.exists():
    print('Copio il dataset in locale, attendi qualche minuto...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Copia completata in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente:', DATASET_DIR)

Copia locale gia presente: /content/dataset_local


## 2. Carico lo split di validazione e risolvo i percorsi

In [ ]:
import pandas as pd

df  = pd.read_csv(SPLIT_CSV)
val = df[df['split'] == 'val'].reset_index(drop=True)
print(f'Val: {len(val)} immagini')
assert val['label'].between(0, NUM_CLASSES-1).all(), 'Etichette fuori range 0..7'

def resolve(fp):
    fp = str(fp)
    for cand in (DATASET_DIR / fp, DRIVE_DATASET_DIR / fp, BASE / fp, Path(fp),
                 DATASET_DIR / Path(fp).name):
        if cand.exists():
            return cand
    return None

val['resolved'] = val['filepath'].map(resolve)
missing = val['resolved'].isna().sum()
assert missing == 0, f'{missing} immagini non risolte.'
samples = list(zip(val['resolved'].tolist(), val['label'].astype(int).tolist()))
print('Esempio percorso risolto:', samples[0][0])


Val: 3102 immagini
Esempio percorso risolto: /content/dataset_local/battery/battery447.jpg


## 3. Le due famiglie di perturbazione

In [ ]:
import io, random
import numpy as np
from PIL import Image, ImageEnhance

LEVELS = {
    'mild':     dict(scale=(0.80,1.00), rot=6,  res=(0.60,0.80), jpeg=(60,80), photo=0.12, hue=0.03, noise=0.0),
    'moderate': dict(scale=(0.60,1.00), rot=12, res=(0.35,0.60), jpeg=(35,60), photo=0.25, hue=0.06, noise=6.0),
}

def _rng_for(index, seed=SEED):
    return random.Random(seed * 1_000_003 + int(index))

# Famiglia A: geometrica / fotometrica
def perturb_geometric(img, index, level=None, seed=SEED):
    p = LEVELS[level or HARD_LEVEL]; r = _rng_for(index, seed)
    W, H = img.size
    s = r.uniform(*p['scale'])
    cw, ch = max(1, int(W*s)), max(1, int(H*s))
    x0 = r.randint(0, W-cw); y0 = r.randint(0, H-ch)
    img = img.crop((x0, y0, x0+cw, y0+ch))
    img = img.rotate(r.uniform(-p['rot'], p['rot']), resample=Image.BILINEAR,
                     expand=False, fillcolor=(128,128,128))
    d = p['photo']
    for Enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = Enh(img).enhance(1.0 + r.uniform(-d, d))
    if p['hue'] > 0:
        hsv = np.asarray(img.convert('HSV')).astype(np.int16)
        hsv[..., 0] = (hsv[..., 0] + int(r.uniform(-p['hue'], p['hue'])*255)) % 256
        img = Image.fromarray(hsv.astype(np.uint8), 'HSV').convert('RGB')
    return img

# Famiglia B: acquisizione
def perturb_acquisition(img, index, level=None, seed=SEED):
    p = LEVELS[level or HARD_LEVEL]; r = _rng_for(index, seed)
    W, H = img.size
    f = r.uniform(*p['res'])
    w2, h2 = max(1, int(W*f)), max(1, int(H*f))
    img = img.resize((w2, h2), Image.BILINEAR).resize((W, H), Image.BILINEAR)
    if p['noise'] > 0:
        arr = np.asarray(img).astype(np.float32)
        rs = np.random.RandomState(seed*7 + int(index))
        arr = np.clip(arr + rs.normal(0, p['noise'], arr.shape), 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    q = r.randint(*p['jpeg'])
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=q); buf.seek(0)
    return Image.open(buf).convert('RGB')

## 4. Preprocessing e cache: clean / geometric / acquisition


In [ ]:
import torch
from torchvision import transforms
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

_preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def _load_rgb(path):
    return Image.open(path).convert('RGB')

N = len(samples)
y       = torch.tensor([lab for _, lab in samples], dtype=torch.long)
clean_X = torch.empty((N, 3, IMG_SIZE, IMG_SIZE), dtype=torch.float16)
geom_X  = torch.empty((N, 3, IMG_SIZE, IMG_SIZE), dtype=torch.float16)
acq_X   = torch.empty((N, 3, IMG_SIZE, IMG_SIZE), dtype=torch.float16)

for i, (path, _) in enumerate(tqdm(samples, desc=f'Preprocess [{HARD_LEVEL}]')):
    img = _load_rgb(path)
    clean_X[i] = _preprocess(img).half()
    geom_X[i]  = _preprocess(perturb_geometric(img, i)).half()
    acq_X[i]   = _preprocess(perturb_acquisition(img, i)).half()

gb = lambda t: round(t.element_size()*t.nelement()/1e9, 2)
print('Cache pronte:', tuple(clean_X.shape), '| RAM ~', gb(clean_X)+gb(geom_X)+gb(acq_X), 'GB')

Device: cuda


Preprocess [mild]:   0%|          | 0/3102 [00:00<?, ?it/s]

/tmp/ipykernel_925/983780577.py:30: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(hsv.astype(np.uint8), 'HSV').convert('RGB')


Cache pronte: (3102, 3, 224, 224) | RAM ~ 2.79 GB


## 5. Valutazione di ogni ricetta su clean / geometric / acquisition

`clean` funge anche da **sanity check**: deve riprodurre i numeri già noti.

In [ ]:
import torch.nn as nn
from torchvision import models
from sklearn.metrics import balanced_accuracy_score, recall_score

def build_resnet18():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def load_ckpt(fname):
    state = torch.load(MODELS_DIR / fname, map_location='cpu')
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m = build_resnet18(); m.load_state_dict(state)
    return m.to(device).eval()

@torch.no_grad()
def predict_all(model, X):
    preds = torch.empty(X.shape[0], dtype=torch.long)
    for s in range(0, X.shape[0], EVAL_BATCH):
        batch = X[s:s+EVAL_BATCH].float().to(device)
        preds[s:s+EVAL_BATCH] = model(batch).argmax(1).cpu()
    return preds.numpy()

def bal_acc(yt, yp):
    return balanced_accuracy_score(yt, yp)

def tpr_per_class(yt, yp):
    return recall_score(yt, yp, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)

METAL   = CLASS_NAMES.index('metal')
PLASTIC = CLASS_NAMES.index('plastic')
y_np    = y.numpy()

rows = []
for name, fname in CANDIDATES.items():
    if not (MODELS_DIR / fname).exists():
        print(f'[skip] {name}: {fname} non trovato'); continue
    model = load_ckpt(fname)
    p_clean = predict_all(model, clean_X)
    p_geom  = predict_all(model, geom_X)
    p_acq   = predict_all(model, acq_X)
    del model; torch.cuda.empty_cache()
    t_acq = tpr_per_class(y_np, p_acq)
    rows.append(dict(
        recipe=name, candidate=name in REAL_CANDIDATES,
        clean=round(bal_acc(y_np, p_clean), 4),
        geometric=round(bal_acc(y_np, p_geom), 4),
        acquisition=round(bal_acc(y_np, p_acq), 4),
        metal_tpr_acq=round(float(t_acq[METAL]), 4),
        plastic_tpr_acq=round(float(t_acq[PLASTIC]), 4),
    ))
    print(f"{name:22s}  clean {rows[-1]['clean']:.4f} | geom {rows[-1]['geometric']:.4f} | acq {rows[-1]['acquisition']:.4f}")

baseline (CE)           clean 0.9666 | geom 0.9449 | acq 0.9584
light                   clean 0.9679 | geom 0.9576 | acq 0.9583
geo_rrc (light+RRC)     clean 0.9574 | geom 0.9591 | acq 0.9452
weighted_loss           clean 0.9606 | geom 0.9470 | acq 0.9582
strong                  clean 0.9540 | geom 0.9449 | acq 0.9548
sampler                 clean 0.9619 | geom 0.9404 | acq 0.9480
geo_rrc+cutmix          clean 0.9649 | geom 0.9607 | acq 0.9537


## 6. Tabella, salvataggio e verdetto sul confound

Il verdetto si legge **solo sulla famiglia Acquisizione** (quella disgiunta da `geo_rrc`).

In [ ]:
import pandas as pd

res = pd.DataFrame(rows)
res.to_csv(RESULTS_DIR / f'confound_families_{HARD_LEVEL}.csv', index=False)
print(res.to_string(index=False)); print()

real = res[res['candidate']].copy()
best_acq = real.sort_values('acquisition', ascending=False).iloc[0]
geo_row  = real[real['recipe'].str.startswith('geo_rrc')].iloc[0]
gap      = round(best_acq['acquisition'] - geo_row['acquisition'], 4)

print('=== Candidati reali, famiglia ACQUISIZIONE (disgiunta da geo_rrc) ===')
print(real[['recipe','geometric','acquisition']]
      .sort_values('acquisition', ascending=False).to_string(index=False))
print()
print(f"Miglior candidato su acquisition: {best_acq['recipe']} ({best_acq['acquisition']:.4f})")
print(f"geo_rrc su acquisition:           {geo_row['acquisition']:.4f}  (Δ dal migliore = {gap:+.4f})")
print()
if gap <= 0.01:
    print(">>> CONFOUND SMONTATO: su perturbazioni MAI viste in training (risoluzione/JPEG/rumore)")
    print(f"    geo_rrc e' in testa o pari (Δ={gap:+.4f} ≤ 0.01). La robustezza e' genuina e trasferisce.")
else:
    print(">>> ATTENZIONE: su acquisition geo_rrc resta indietro di", f"{gap:.4f}", "(> 0.01).")
    print("    Parte del vantaggio in hard-val era allineamento col suo augmentation. Rivedere la selezione")
    print("    (clean-val, degrado a RealWaste, efficienza come criteri alternativi).")

             recipe  candidate  clean  geometric  acquisition  metal_tpr_acq  plastic_tpr_acq
      baseline (CE)      False 0.9666     0.9449       0.9584         0.9221           0.8902
              light       True 0.9679     0.9576       0.9583         0.9156           0.8960
geo_rrc (light+RRC)       True 0.9574     0.9591       0.9452         0.8636           0.8613
      weighted_loss       True 0.9606     0.9470       0.9582         0.9221           0.8960
             strong      False 0.9540     0.9449       0.9548         0.9286           0.8439
            sampler      False 0.9619     0.9404       0.9480         0.8831           0.8786
     geo_rrc+cutmix      False 0.9649     0.9607       0.9537         0.8766           0.8844

=== Candidati reali, famiglia ACQUISIZIONE (disgiunta da geo_rrc) ===
             recipe  geometric  acquisition
              light     0.9576       0.9583
      weighted_loss     0.9470       0.9582
geo_rrc (light+RRC)     0.9591       0.9452

